# Whisper Pro Ultra-Fast: UI Limpia y Descarga Automática

Este cuaderno utiliza **Faster-Whisper** con barras de progreso en tiempo real. Al finalizar, empaqueta todas las transcripciones en un archivo ZIP para su descarga.

In [ ]:
# 1. Instalación de dependencias optimizadas
!pip install -q faster-whisper tqdm
!apt-get install -y -q ffmpeg

In [ ]:
import os
import datetime
import zipfile
import torch
from pathlib import Path
from tqdm.auto import tqdm
from faster_whisper import WhisperModel
from IPython.display import FileLink, display

# --- CONFIGURACIÓN ---
INPUT_DIR = "/kaggle/input/datasets/danieldobles/test-whisper"
OUTPUT_DIR = "/kaggle/working/transcriptions"
ZIP_NAME = "transcripciones_completas.zip"
MODEL_NAME = "large-v3"
AUDIO_EXTENSIONS = (".mp3", ".wav", ".m4a", ".flac", ".ogg", ".mp4")

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Verificación de GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cpu":
    print("⚠ ATENCIÓN: No se detectó GPU. El proceso será LENTO.")
else:
    print(f"✔ GPU Detectada: {torch.cuda.get_device_name(0)}")

print(f"Cargando modelo {MODEL_NAME}...")
model = WhisperModel(MODEL_NAME, device=device, compute_type="float16" if device=="cuda" else "int8")

In [ ]:
def transcribe_with_progress(file_path):
    file_path = Path(file_path)
    output_base = Path(OUTPUT_DIR) / file_path.stem
    
    if os.path.exists(f"{output_base}.txt"):
        return f"[OMITIDO] {file_path.name}"

    segments, info = model.transcribe(
        str(file_path), 
        beam_size=5, 
        language="es", 
        vad_filter=True
    )

    full_text = []
    # Barra de progreso basada en la duración del audio (info.duration)
    with tqdm(total=info.duration, unit="sec", desc=file_path.name[:20], leave=False) as pbar:
        for segment in segments:
            full_text.append(segment.text)
            pbar.update(segment.end - pbar.n) # Actualiza la barra según los segundos procesados

    final_content = " ".join(full_text).strip()
    
    # Guardar archivos
    with open(f"{output_base}.txt", "w", encoding="utf-8") as f: f.write(final_content)
    with open(f"{output_base}.md", "w", encoding="utf-8") as f: f.write(f"# {file_path.name}\n\n{final_content}")
    
    return f"✔ Completado: {file_path.name}"

# --- PROCESAMIENTO ---
files = [os.path.join(r, f) for r, d, fs in os.walk(INPUT_DIR) for f in fs if f.lower().endswith(AUDIO_EXTENSIONS)]
print(f"\nProcesando {len(files)} archivos...")

for f in files:
    status = transcribe_with_progress(f)
    print(status)

# --- EMPAQUETADO ZIP ---
print("\nGenerando archivo comprimido...")
with zipfile.ZipFile(ZIP_NAME, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(OUTPUT_DIR):
        for file in files:
            zipf.write(os.path.join(root, file), arcname=file)

print(f"\n--- PROCESO FINALIZADO ---")
display(FileLink(ZIP_NAME, result_html_prefix="Haz clic aquí para descargar todas las transcripciones: "))